# NYC Mobility - Source Inspection

## Why we inspect first

We check the source first so we know what we are working with before we create tables. These checks describe the raw files only; they do not clean or change any records.


## Green Taxi

The required monthly Parquet files cover March, April, and May 2026. We start with a temporary March view so the source can be inspected without creating a Bronze object.

### March preview


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW src_green_taxi_2026_03 AS
SELECT *
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
    format => 'parquet'
);

The temporary view points to the landed March file and exists only for this notebook session.


In [0]:
%sql
SELECT *
FROM src_green_taxi_2026_03
LIMIT 20;

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,_rescued_data
2,2026-03-01T00:46:53.000,2026-03-01T00:48:34.000,N,5,82,129,1,0.26,50.0,0.0,0.0,1.0,0.0,null,1.0,52.0,1,2,0.0,0.0,null
2,2026-03-01T00:06:37.000,2026-03-01T00:38:51.000,N,5,97,100,1,8.22,75.0,0.0,0.0,0.06,0.0,null,1.0,79.56,1,2,2.75,0.75,null
2,2026-03-01T00:04:05.000,2026-03-01T00:30:26.000,N,1,66,237,1,8.4,36.6,1.0,0.5,5.0,0.0,null,1.0,47.6,1,1,2.75,0.75,null
2,2026-03-01T00:32:52.000,2026-03-01T00:50:06.000,N,1,130,130,1,2.84,17.7,1.0,0.5,0.0,0.0,null,1.0,20.2,2,1,0.0,0.0,null
2,2026-03-01T00:22:30.000,2026-03-01T00:27:33.000,N,1,166,41,1,0.83,7.2,1.0,0.5,1.94,0.0,null,1.0,11.64,1,1,0.0,0.0,null
2,2026-03-01T00:38:03.000,2026-03-01T00:49:18.000,N,1,130,121,1,3.03,14.9,1.0,0.5,3.48,0.0,null,1.0,20.88,1,1,0.0,0.0,null
1,2026-03-01T00:35:34.000,2026-03-01T00:35:59.000,N,5,255,255,1,4.4,80.0,0.0,0.0,0.0,0.0,null,0.0,80.0,3,2,0.0,0.0,null
2,2026-03-01T00:07:14.000,2026-03-01T00:16:30.000,N,1,42,69,1,1.74,11.4,1.0,0.5,0.0,0.0,null,1.0,13.9,2,1,0.0,0.0,null
2,2026-03-01T00:43:35.000,2026-03-01T01:03:40.000,N,1,244,213,1,8.16,35.2,1.0,0.5,0.0,0.0,null,1.0,37.7,2,1,0.0,0.0,null
2,2026-03-01T00:33:45.000,2026-03-01T00:52:51.000,N,1,75,235,1,6.06,27.5,1.0,0.5,0.08,0.0,null,1.0,30.08,1,1,0.0,0.0,null


### Schema

We inspect the inferred source columns and data types before designing later layers.


In [0]:
%sql
DESCRIBE src_green_taxi_2026_03;

col_name,data_type,comment
VendorID,int,null
lpep_pickup_datetime,timestamp_ntz,null
lpep_dropoff_datetime,timestamp_ntz,null
store_and_fwd_flag,string,null
RatecodeID,bigint,null
PULocationID,int,null
DOLocationID,int,null
passenger_count,bigint,null
trip_distance,double,null
fare_amount,double,null


### March row count

The original run recorded **44,208 source rows** for March.


In [0]:
%sql
SELECT COUNT(*) AS march_source_row_count
FROM src_green_taxi_2026_03;

march_source_row_count
44208


### Basic source quality checks

We check timestamps, location IDs, negative numeric values, and rescued data. These are observations only—Bronze must still keep the rows as received.


In [0]:
%sql
WITH source AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    COUNT(*) AS total_rows,

    MIN(lpep_pickup_datetime) AS earliest_pickup,
    MAX(lpep_pickup_datetime) AS latest_pickup,

    SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
    SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,

    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,

    SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,

    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,

    SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows

FROM source;

total_rows,earliest_pickup,latest_pickup,null_pickup_datetime,null_dropoff_datetime,null_pickup_location,null_dropoff_location,invalid_time_order,negative_trip_distance,negative_fare_amount,negative_total_amount,rescued_data_rows
44208,2009-01-01T01:35:31.000,2026-03-31T23:57:29.000,0,0,0,0,1,0,111,113,0


### Observation

March has no null pickup/drop-off timestamps or location IDs and no rescued rows. We did observe one invalid time order, 111 negative fares, 113 negative totals, and an earliest pickup in 2009.

### Why it matters

These records may need a business rule later, but changing them now would stop Bronze from being raw.

### Decision

Keep every row in Bronze and revisit the unusual records in Silver.


### Observed date overlap and unusual records

This check shows which pickup year and month appear inside the March file.


In [0]:
%sql
WITH source AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    YEAR(lpep_pickup_datetime) AS pickup_year,
    MONTH(lpep_pickup_datetime) AS pickup_month,
    COUNT(*) AS row_count
FROM source
GROUP BY
    YEAR(lpep_pickup_datetime),
    MONTH(lpep_pickup_datetime)
ORDER BY
    pickup_year,
    pickup_month;

pickup_year,pickup_month,row_count
2009,1,1
2026,2,8
2026,3,44199


The March file contains 44,199 March 2026 rows, eight February 2026 rows, and one January 2009 row. We record the overlap instead of filtering it here.


### March-May monthly profiling

We run the same basic checks across all three landed files.


In [0]:
%sql
-- profile all three monthly source files before bronze

WITH march AS (
    SELECT
        '2026-03' AS source_month,
        COUNT(*) AS row_count,
        MIN(lpep_pickup_datetime) AS earliest_pickup,
        MAX(lpep_pickup_datetime) AS latest_pickup,
        SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
        SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,
        SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
        SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
),

april AS (
    SELECT
        '2026-04' AS source_month,
        COUNT(*) AS row_count,
        MIN(lpep_pickup_datetime) AS earliest_pickup,
        MAX(lpep_pickup_datetime) AS latest_pickup,
        SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
        SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,
        SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
        SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
        format => 'parquet'
    )
),

may AS (
    SELECT
        '2026-05' AS source_month,
        COUNT(*) AS row_count,
        MIN(lpep_pickup_datetime) AS earliest_pickup,
        MAX(lpep_pickup_datetime) AS latest_pickup,
        SUM(CASE WHEN lpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
        SUM(CASE WHEN lpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pickup_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_dropoff_location,
        SUM(CASE WHEN lpep_dropoff_datetime < lpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_time_order,
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
        SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_data_rows
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
        format => 'parquet'
    )
)

SELECT * FROM march
UNION ALL
SELECT * FROM april
UNION ALL
SELECT * FROM may
ORDER BY source_month;

source_month,row_count,earliest_pickup,latest_pickup,null_pickup_datetime,null_dropoff_datetime,null_pickup_location,null_dropoff_location,invalid_time_order,negative_trip_distance,negative_fare_amount,negative_total_amount,rescued_data_rows
2026-03,44208,2009-01-01T01:35:31.000,2026-03-31T23:57:29.000,0,0,0,0,1,0,111,113,0
2026-04,44238,2026-03-31T23:28:50.000,2026-05-01T07:53:18.000,0,0,0,0,0,0,153,155,0
2026-05,44921,2008-12-31T23:05:50.000,2026-05-31T23:59:13.000,0,0,0,0,0,0,120,123,0


The source counts are **44,208 for March**, **44,238 for April**, and **44,921 for May**. Negative fare and total values remain present, while the inspected key timestamps and location IDs have no nulls.


In [0]:
%sql
WITH all_taxi AS (
    SELECT '2026-03' AS source_month, *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT '2026-04' AS source_month, *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT '2026-05' AS source_month, *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
        format => 'parquet'
    )
)

SELECT
    source_month,
    YEAR(lpep_pickup_datetime) AS pickup_year,
    MONTH(lpep_pickup_datetime) AS pickup_month,
    COUNT(*) AS row_count
FROM all_taxi
GROUP BY
    source_month,
    YEAR(lpep_pickup_datetime),
    MONTH(lpep_pickup_datetime)
ORDER BY
    source_month,
    pickup_year,
    pickup_month;

source_month,pickup_year,pickup_month,row_count
2026-03,2009,1,1
2026-03,2026,2,8
2026-03,2026,3,44199
2026-04,2026,3,1
2026-04,2026,4,44235
2026-04,2026,5,2
2026-05,2008,12,2
2026-05,2026,4,8
2026-05,2026,5,44911


### Observation

The file labels and pickup months do not align perfectly. April includes one March and two May pickups; May includes two December 2008 and eight April pickups.

### Decision

Treat the filename as ingestion provenance and preserve the source timestamps. Any date-quality rule belongs in Silver.


## Taxi Zones

We inspect the reference CSV before using it to interpret pickup and drop-off IDs.


In [0]:
%sql
-- temporary source inspection only
CREATE OR REPLACE TEMP VIEW src_taxi_zones AS

SELECT *
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv',
    format => 'csv',
    header => true
);

### Preview


In [0]:
%sql
SELECT *
FROM src_taxi_zones
LIMIT 20;

LocationID,Borough,Zone,service_zone,_rescued_data
1,EWR,Newark Airport,EWR,null
2,Queens,Jamaica Bay,Boro Zone,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,null
4,Manhattan,Alphabet City,Yellow Zone,null
5,Staten Island,Arden Heights,Boro Zone,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,null
7,Queens,Astoria,Boro Zone,null
8,Queens,Astoria Park,Boro Zone,null
9,Queens,Auburndale,Boro Zone,null
10,Queens,Baisley Park,Boro Zone,null


### Schema


In [0]:
%sql
DESCRIBE src_taxi_zones;

col_name,data_type,comment
LocationID,int,null
Borough,string,null
Zone,string,null
service_zone,string,null
_rescued_data,string,null


### Row count

The original inspection recorded **265 Taxi Zone rows**.


In [0]:
%sql
SELECT COUNT(*) AS taxi_zone_row_count
FROM src_taxi_zones;

taxi_zone_row_count
265


### LocationID uniqueness

An empty result means no duplicate `LocationID` was found.


In [0]:
%sql
SELECT
    LocationID,
    COUNT(*) AS record_count
FROM src_taxi_zones
GROUP BY LocationID
HAVING COUNT(*) > 1;

LocationID,record_count


### Required-field null checks

The original result shows zero nulls in `LocationID`, `Borough`, `Zone`, and `service_zone`.


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END) AS null_location_id,
    SUM(CASE WHEN Borough IS NULL THEN 1 ELSE 0 END) AS null_borough,
    SUM(CASE WHEN Zone IS NULL THEN 1 ELSE 0 END) AS null_zone,
    SUM(CASE WHEN service_zone IS NULL THEN 1 ELSE 0 END) AS null_service_zone
FROM src_taxi_zones;

total_rows,null_location_id,null_borough,null_zone,null_service_zone
265,0,0,0,0


### Pickup relationship check

An empty result means every March pickup location matched the lookup.


In [0]:
%sql
-- check pickup location ids that do not exist in the taxi zone lookup

WITH taxi AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    t.PULocationID,
    COUNT(*) AS trip_count
FROM taxi t
LEFT JOIN src_taxi_zones z
    ON t.PULocationID = z.LocationID
WHERE z.LocationID IS NULL
GROUP BY t.PULocationID
ORDER BY trip_count DESC;

PULocationID,trip_count


### Drop-off relationship check

An empty result means every March drop-off location matched the lookup.


In [0]:
%sql
-- check dropoff location ids that do not exist in the taxi zone lookup

WITH taxi AS (
    SELECT *
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )
)

SELECT
    t.DOLocationID,
    COUNT(*) AS trip_count
FROM taxi t
LEFT JOIN src_taxi_zones z
    ON t.DOLocationID = z.LocationID
WHERE z.LocationID IS NULL
GROUP BY t.DOLocationID
ORDER BY trip_count DESC;

DOLocationID,trip_count


We found 265 Taxi Zones, no duplicate `LocationID`, no required-field nulls, and no unmatched March pickup or drop-off IDs. This is verified source evidence. The Bronze implementation is now prepared in the Bronze notebooks, but its new results remain pending until those cells are run in Databricks.


## Weather

Run the ingestion notebook first so the raw Open-Meteo JSON exists in R2. Here we only inspect the saved file; we do not flatten or clean it.

### Request and saved-file metadata

- Source system: `open_meteo`
- API: Open-Meteo Archive API
- Requested period: `2026-03-01` through `2026-05-31`
- Requested timezone: `America/New_York`
- Exact saved file: `open_meteo_2026-03-01_2026-05-31.json`
- Verified ingestion response: HTTP 200


In [0]:
%python
from pathlib import Path

weather_file = Path(
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/weather/"
    "open_meteo_2026-03-01_2026-05-31.json"
)

print("Exists:", weather_file.exists())
print("Size:", weather_file.stat().st_size, "bytes")

Exists: True
Size: 100500 bytes


### JSON structure and metadata


In [0]:
%python
import json

weather_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/weather/"
    "open_meteo_2026-03-01_2026-05-31.json"
)

with open(weather_path, "r") as file:
    weather = json.load(file)

print("Top-level keys:")
print(weather.keys())

print("\nLocation metadata:")
print("Latitude:", weather.get("latitude"))
print("Longitude:", weather.get("longitude"))
print("Timezone:", weather.get("timezone"))
print("UTC offset seconds:", weather.get("utc_offset_seconds"))

print("\nHourly variables:")
print(weather["hourly"].keys())

times = weather["hourly"]["time"]

print("\nHourly record check:")
print("Number of timestamps:", len(times))
print("First timestamp:", times[0])
print("Last timestamp:", times[-1])

Top-level keys:
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

Location metadata:
Latitude: 40.738136
Longitude: -74.04254
Timezone: America/New_York
UTC offset seconds: -14400

Hourly variables:
dict_keys(['time', 'temperature_2m', 'precipitation', 'rain', 'snowfall', 'weather_code', 'wind_speed_10m'])

Hourly record check:
Number of timestamps: 2208
First timestamp: 2026-03-01T00:00
Last timestamp: 2026-05-31T23:00


The request returned HTTP 200 during ingestion. The saved response contains 2,208 hourly timestamps from `2026-03-01T00:00` through `2026-05-31T23:00`. The API returned nearby grid coordinates, which we retain as source metadata.


In [0]:
hourly = weather["hourly"]

for column_name, values in hourly.items():
    print(column_name, len(values))

time 2208
temperature_2m 2208
precipitation 2208
rain 2208
snowfall 2208
weather_code 2208
wind_speed_10m 2208


### Hourly-array null checks

Each inspected hourly array has 2,208 values.


In [0]:
for column_name, values in hourly.items():
    null_count = sum(value is None for value in values)

    print(
        column_name,
        "| records:", len(values),
        "| nulls:", null_count
    )

time | records: 2208 | nulls: 0
temperature_2m | records: 2208 | nulls: 0
precipitation | records: 2208 | nulls: 0
rain | records: 2208 | nulls: 0
snowfall | records: 2208 | nulls: 0
weather_code | records: 2208 | nulls: 0
wind_speed_10m | records: 2208 | nulls: 0


The existing output shows zero nulls for time, temperature, precipitation, rain, snowfall, weather code, and wind speed. The 2,208 observations remain inside the raw JSON. The one-record Bronze implementation is prepared, but it has not been executed or validated in Databricks yet.


## Traffic Advisory

This is our optional web-scraping source. The original run saved one raw HTML file and one metadata JSON file. We point to those verified files for inspection.


In [ ]:
from pathlib import Path

html_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory/"
    "nyc_dot_weekend_traffic_20260914T040846Z.html"
)
metadata_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory/"
    "nyc_dot_weekend_traffic_20260914T040846Z.metadata.json"
)


### Saved-file confirmation


In [0]:
from pathlib import Path

print("HTML exists:", Path(html_path).exists())
print("Metadata exists:", Path(metadata_path).exists())

print("HTML size:", Path(html_path).stat().st_size, "bytes")
print("Metadata size:", Path(metadata_path).stat().st_size, "bytes")

HTML exists: True
Metadata exists: True
HTML size: 53391 bytes
Metadata size: 295 bytes


### Raw HTML preview


In [0]:
with open(html_path, "r", encoding="utf-8", errors="replace") as file:
    preview = file.read(1000)

print(preview)

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
                "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">


<html dir="ltr" lang="en-US">

<head>

<script>
if (document.location.host == "www1.nyc.gov") {
                document.location.href = (document.location.href).replace("www1.nyc.gov","www.nyc.gov");
}
</script>




<meta http-equiv="Content-type" content="text/html; charset=utf-8" />
<meta http-equiv="X-UA-Compatible" content="IE=10"/>
<meta http-equiv="expires" content="-1" />
<meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT" />




  <meta charset="utf-8">
  <meta name="referrer" content="no-referrer-when-downgrade">
  <meta name="viewport" content="width=device-width, initial-scale=1"/>
  <meta name="google-translate-customization" content="4707bd7f535893a0-45bca7b6a97e5a2d-g609df9381571b349-c"/>
  <link rel="stylesheet" href="../../css/style.min.css">
  <!--[if lt IE 9]><link rel="stylesheet" href="../../css/ie.min.css"><![endif]-->


<ti

### Page title and headings


In [0]:
from bs4 import BeautifulSoup

with open(html_path, "r", encoding="utf-8", errors="replace") as file:
    soup = BeautifulSoup(file, "html.parser")

print("Page title:")
print(soup.title.get_text(" ", strip=True) if soup.title else "No title found")

print("\nHeadings found:")
for heading in soup.find_all(["h1", "h2", "h3", "h4"]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(heading.name, "|", text)

Page title:
NYC DOT Weekend Traffic Advisory

Headings found:
h1 | Weekend Traffic Advisory
h2 | Additional Resources
h2 | Weekend Traffic Advisory for Friday September 11, 2026, to Sunday September 13, 2026
h2 | Manhattan
h3 | Festivals, Parades and Events
h4 | New York Fashion Week
h4 | September 11 Commemoration Ceremony
h4 | 8th Avenue Fall Fair
h4 | The 24th Autumn Moon Cultural Festival and Children’s Lantern Parade
h4 | NYRR New Balance 5th Avenue Mile 2026
h2 | Bronx
h2 | Brooklyn
h3 | Festivals, Parades and Events
h4 | Mexican Independence Parade
h2 | Queens
h3 | Festivals, Parades and Events
h4 | US Open


### Body-text inspection


In [0]:
page_text = soup.get_text("\n", strip=True)

print(page_text[:5000])

NYC DOT Weekend Traffic Advisory
Skip to main content
NYC
NYC Resources
311
Office of the Mayor
Motorist &
        Parking
Weekend Traffic Advisory
Each week NYC DOT posts a list of streets, bridges, highways, and tunnels that are temporarily closed for planned
  construction activities, special events, parades, and other events. A preliminary advisory is posted early in the week
  and is updated when the final advisory is posted on Friday. This information is subject to change including all event
  dates, times, and routes, and does not reflect closures due to emergency situations nor long-term construction
  projects.
Additional Resources
Subscribe to weekend traffic advisory email
    updates
Check NYC DOT’s Weekly Traffic Advisory
Check NYC DOT’s special traffic advisories
Visit the NYC Street Closures Map
Visit
NYC Office of Emergency Management
and
NYPD
for emergency street closure updates
Visit 511NY for New York State's official traffic and travel information
Other state and lo

### Table check

The current page contains no HTML tables, so any later parser must rely on the observed heading and paragraph structure.


In [0]:
tables = soup.find_all("table")

print("Number of HTML tables:", len(tables))

Number of HTML tables: 0


### Keyword checks


In [0]:
keywords = [
    "September",
    "2026",
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Bronx",
    "Staten Island"
]

for keyword in keywords:
    print(keyword, "->", keyword.lower() in page_text.lower())

September -> True
2026 -> True
Manhattan -> True
Brooklyn -> True
Queens -> True
Bronx -> True
Staten Island -> True


### Event-heading structure


In [0]:
# inspect how advisory/event headings are structured in the raw html

event_headings = soup.find_all("h4")

print("Number of h4 event headings:", len(event_headings))

for heading in event_headings[:10]:
    print("\nEVENT:")
    print(heading.get_text(" ", strip=True))

    next_element = heading.find_next_sibling()

    if next_element:
        print("NEXT TAG:", next_element.name)
        print(
            "DETAIL:",
            next_element.get_text(" ", strip=True)[:500]
        )
    else:
        print("No next sibling found")

Number of h4 event headings: 7

EVENT:
New York Fashion Week
NEXT TAG: p
DETAIL: The following streets will be closed Thursday September 10th, 2026 to Tuesday September 15th, 2026, for the New York Fashion Week event at the Discretion of the NYPD in Manhattan

EVENT:
September 11 Commemoration Ceremony
NEXT TAG: p
DETAIL: The following streets will be closed Friday September 11th, 2026, for the September 11 Commemoration Ceremony event at the Discretion of the NYPD in Manhattan

EVENT:
8th Avenue Fall Fair
NEXT TAG: p
DETAIL: The following streets will be closed Saturday September 12th, 2026, for the 8th Avenue Fall Fair event at the Discretion of the NYPD in Manhattan

EVENT:
The 24th Autumn Moon Cultural Festival and Children’s Lantern Parade
NEXT TAG: p
DETAIL: The following streets will be closed Saturday September 12th, 2026, for the 24th Autumn Moon Cultural festival and Children’s Lantern Parade event at the Discretion of the NYPD in Manhattan

EVENT:
NYRR New Balance 5th Avenue

### Observation

The saved advisory describes September 2026 events, while the Taxi analytical period is March-May 2026.

### Why it matters

Joining these records would be temporally incorrect.

### Decision

Keep this as a bonus ingestion and source-inspection demonstration only. Do not integrate it into March-May analytics.


## Source Inspection Summary

- Green Taxi: three monthly Parquet files profiled; counts verified; unusual dates and negative amounts documented for later Silver decisions.
- Taxi Zones: 265 rows; unique `LocationID`; required fields complete; March pickup and drop-off IDs matched.
- Weather: raw JSON saved and structurally inspected; 2,208 hourly values per field with zero observed nulls.
- Traffic Advisory: raw HTML and metadata saved and inspected as a September 2026 bonus example.

Green Taxi already has executed Bronze evidence. The Taxi Zones, Weather, and bonus Traffic Advisory Bronze implementations are prepared in the next notebooks. Their results must be produced in Databricks before we call them validated.
